# Baseline — TabPFN

Foundation model pré-treinado para dados tabulares pequenos ("treina" por in-context learning, sem gradiente). Mesmos dados/split/CV do `data_prep` e `leave_one_property_out`, via `sk_eval` — comparação justa com os demais.

**Requer licença aceita uma vez** (não tem terminal interativo aqui):
1. https://ux.priorlabs.ai → login/cadastro → aba "Licenses" → aceitar.
2. Copiar a API key em https://ux.priorlabs.ai/account.
3. Definir `TABPFN_TOKEN` (célula abaixo, ou `export TABPFN_TOKEN=...` antes de abrir o notebook).

## 1. Setup

In [ ]:
import os, sys
while not os.path.isdir("support_scripts"):
    os.chdir("..")
sys.path.append("support_scripts")

# só necessário se ainda não tiver sido feito no ambiente (ex.: `export TABPFN_TOKEN=...`)
# os.environ["TABPFN_TOKEN"] = "cole-sua-key-aqui"

import pandas as pd
import matplotlib.pyplot as plt
from tabpfn import TabPFNRegressor
import data_prep as dp
import sk_eval

## 2. Dados

In [ ]:
d = dp.get_data()
df, groups = d["df"], d["groups"]
print(f"{len(df)} animais | {len(d['feature_cols'])} preditores")
print("grupos:", groups.value_counts().to_dict())

## 3. CV + LOPO (via `sk_eval`)

TabPFN aceita as features cruas (não escaladas); `scale=False`. Sem grid search — o modelo é pré-treinado, não há hiperparâmetro de arquitetura para tunar aqui.

In [ ]:
make_model = lambda: TabPFNRegressor(random_state=dp.SEED)
res = sk_eval.registrar("TabPFN", make_model, scale=False, d=d)
res["sem_jitter"].agg(["mean", "std"]).round(4)

In [ ]:
print("LOPO (treina 1 fazenda, testa a outra):")
res["lopo"]

## 4. Predito × real (OOF)

In [ ]:
y_true = df[dp.TARGET].values
y_oof = sk_eval.oof(d, make_model, scale=False)
r2_oof = dp.metrics(y_true, y_oof)["R2"]

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(y_true, y_oof, alpha=0.6, edgecolor="k", linewidth=.3)
lims = [min(y_true.min(), y_oof.min()), max(y_true.max(), y_oof.max())]
ax.plot(lims, lims, "r--", lw=1)
ax.set_xlabel("GMD real (kg/dia)"); ax.set_ylabel("GMD predito (OOF)")
ax.set_title(f"TabPFN — predito vs real (R² OOF = {r2_oof:.3f})")
plt.tight_layout(); plt.show()

## 5. Notas

- Mesmo `get_cv()`/`leave_one_property_out` dos demais — comparação direta em `results/model_comparison.csv`.
- Primeira execução baixa os pesos do modelo (HuggingFace) — precisa de internet e da licença aceita (ver topo).